# [실습] Advanced RAG

RAG의 기본 베이스 체인에서 시작하여, 다양한 기능을 추가해 보겠습니다.   

### 라이브러리 설치  

랭체인 관련 라이브러리와 벡터 데이터베이스 라이브러리를 설치합니다.   

In [ ]:
!pip install jsonlines openai langchain==0.3.27 langchain-openai langchain-community==0.3.27 langchain-chroma tiktoken rank_bm25 pymupdf kiwipiepy

     ---------------------------------------- 0.0/35.5 MB ? eta -:--:--
     ----- ---------------------------------- 5.2/35.5 MB 27.8 MB/s eta 0:00:02
     ---------- ---------------------------- 10.0/35.5 MB 24.8 MB/s eta 0:00:02
     ---------------- ---------------------- 14.9/35.5 MB 25.1 MB/s eta 0:00:01
     ----------------------- --------------- 21.0/35.5 MB 26.2 MB/s eta 0:00:01
     ----------------------------- --------- 27.0/35.5 MB 26.9 MB/s eta 0:00:01
     ----------------------------------- --- 32.5/35.5 MB 27.0 MB/s eta 0:00:01
     --------------------------------------  35.4/35.5 MB 26.5 MB/s eta 0:00:01
     --------------------------------------- 35.5/35.5 MB 25.1 MB/s eta 0:00:00
  Preparing metadata (setup.py): started
  Preparing metadata (setup.py): finished with status 'done'
   ---------------------------------------- 0.0/2.4 MB ? eta -:--:--
   ---------------------------------------- 2.4/2.4 MB 20.5 MB/s eta 0:00:00
  Created wheel for kiwipiepy_model: fil

  DEPRECATION: Building 'kiwipiepy_model' using the legacy setup.py bdist_wheel mechanism, which will be removed in a future version. pip 25.3 will enforce this behaviour change. A possible replacement is to use the standardized build interface by setting the `--use-pep517` option, (possibly combined with `--no-build-isolation`), or adding a `pyproject.toml` file to the source tree of 'kiwipiepy_model'. Discussion can be found at https://github.com/pypa/pip/issues/6334


### LLM과 임베딩 모델 불러오기

In [33]:
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_openai import OpenAIEmbeddings

load_dotenv('.env', override=True)
if os.environ.get('OPENAI_API_KEY'):
    print('OpenAI API 키 확인')

llm = ChatOpenAI(model="gpt-4.1-mini", temperature = 0)
sllm = ChatOpenAI(model='gpt-4.1-nano', temperature = 0.7)
reasoning_llm = ChatOpenAI(model='gpt-5-mini')

embeddings = OpenAIEmbeddings(model = 'text-embedding-3-large', chunk_size=100)

OpenAI API 키 확인


reports.zip 파일을 압축 해제합니다.

In [4]:
import zipfile

with zipfile.ZipFile('reports.zip', 'r') as zip_ref:
    zip_ref.extractall('.')

## 데이터 불러오기

폴더에 포함된 문서들을 pdf 로더로 불러옵니다.   
glob 라이브러리를 사용합니다.

In [ ]:
from langchain_core.documents import Document
from glob import glob

# 모든 PDF 파일을 glob으로 찾음
pdf_files = glob("reports/*.pdf")
pdf_files

['reports\\반기보고서(2024.08.14).pdf',
 'reports\\분기보고서(2024.11.14).pdf',
 'reports\\사업보고서(2025.03.11).pdf']

In [6]:
from langchain_community.document_loaders import PyMuPDFLoader

# 각 PDF 파일에서 페이지별로 내용을 불러와 하나로 합침
all_papers=[]

for i, path_paper in enumerate(pdf_files):
    loader = PyMuPDFLoader(path_paper)
    pages = loader.load()
    # 문서가 주어지면 Document List를 출력
    # Page별 저장 --> 청킹을 다시 수행

    doc = Document(page_content='', metadata = {'index':i, 'source':pages[0].metadata['source']})
    for page in pages:
        doc.page_content += page.page_content +' '

    doc.page_content = doc.page_content.replace('\n', ' ')
    for _ in range(10):
        doc.page_content = doc.page_content.replace('  ', ' ')
        doc.page_content = doc.page_content.replace('..', '.')

    all_papers.append(doc)

print(len(all_papers))
all_papers[0].page_content[:1000]

3


'목 차 반 기 보 고 서.1 【 대표이사 등의 확인 】.2 I. 회사의 개요.3 1. 회사의 개요.3 2. 회사의 연혁.5 3. 자본금 변동사항.8 4. 주식의 총수 등.9 5. 정관에 관한 사항.10 II. 사업의 내용.12 1. 사업의 개요.12 2. 주요 제품 및 서비스.13 3. 원재료 및 생산설비.17 4. 매출 및 수주상황.19 5. 위험관리 및 파생거래.20 6. 주요계약 및 연구개발활동.22 7. 기타 참고사항.25 III. 재무에 관한 사항.34 1. 요약재무정보.34 2. 연결재무제표.37 2-1. 연결 재무상태표.37 2-2. 연결 손익계산서.38 2-3. 연결 포괄손익계산서.39 2-4. 연결 자본변동표.39 2-5. 연결 현금흐름표.40 3. 연결재무제표 주석.42 1. 지배기업의 개요 (연결) .42 2. 연결재무제표 작성기준 및 중요한 회계정책 (연결).42 3. 중요한 판단과 추정 불확실성의 주요 원천 (연결) .43 4. 영업부문 (연결) .44 5. 범주별 금융상품 (연결) .46 6. 공정가치 (연결) .48 7. 공정가치측정금융자산 (연결).51 8. 관계기업투자주식 (연결).54 9. 종속기업 (연결) .58 10. 유형자산 (연결) .67 11. 무형자산 (연결) .68 12. 투자부동산 (연결).69 13. 리스 (연결) .71 14. 리스부채 (연결) .73 15. 금융리스채권 (연결).74 16. 충당부채 (연결) .76 17. 재무위험관리 (연결).79 18. 우발채무와 약정사항 (연결) .81 19. 납입자본 (연결) .83 20. 이익잉여금 (연결).84 21. 기타자본항목 (연결).85 22. 매출액 (연결).86 23. 판매비와 관리비 (연결).89 24. 기타수익 및 기타비용 (연결).91 25. 금융수익 및 금융비용 (연결).92 26. 법인세비용 (연결).93 27. 현금흐름표 (연결).94 28. 주당이익 (연결) .96 29. 특수관계자 (연결).97 30. 주식기준보상 (연결).109 4. 재무

글자 수와 토큰 수를 확인해 보겠습니다.

In [10]:
import tiktoken

encoder = tiktoken.encoding_for_model('gpt-4.1-mini') # 4.1, 5, 4o 모두 동일
for paper in all_papers:
    print(len(paper.page_content), len(encoder.encode(paper.page_content)), paper.metadata['source'])

345728 182859 reports\반기보고서(2024.08.14).pdf
212976 123712 reports\분기보고서(2024.11.14).pdf
510249 283113 reports\사업보고서(2025.03.11).pdf


# 토큰 단위로 청킹하기   

tiktoken이나 huggingface를 이용해 토큰 단위 청킹을 수행합니다.

In [11]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
import tiktoken
token_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    model_name="gpt-4.1-mini",
    chunk_size=2000,
    chunk_overlap=400,
)
# from_huggingface_tokenizer : 허깅페이스 모델에서 토크나이저 가져오기

token_chunks = token_splitter.split_documents(all_papers)
print(len(token_chunks))

371


벡터 DB를 구성합니다.   

In [14]:
from langchain_chroma import Chroma
from tqdm import tqdm

Chroma().delete_collection()
db = Chroma(embedding_function=embeddings,
                           persist_directory="./chroma2",
                           collection_metadata={'hnsw:space':'l2'},
                           collection_name='finance',
                           )

db.add_documents(token_chunks)

retriever = db.as_retriever(search_kwargs={"k": 5})

# filter 옵션을 통해 특정 메타데이터를 가진 벡터만 검색 가능
# retriever = db.as_retriever(search_kwargs={"k": 5,"filter":{'author':'Hyungho Byun'}})

RAG 체인을 구현합니다.

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate([
    ("user", '''당신은 QA(Question-Answering)을 수행하는 Assistant입니다.
다음의 Context를 이용하여 Question에 한국어로 답변하세요.
정확한 답변을 제공하세요.
만약 모든 Context를 다 확인해도 정보가 없다면, "정보가 부족하여 답변할 수 없습니다."를 출력하세요.
---
Context: {context}
---
Question: {question}''')])
prompt.pretty_print()

================================ Human Message =================================

당신은 QA(Question-Answering)을 수행하는 Assistant입니다.
다음의 Context를 이용하여 Question에 한국어로 답변하세요.
정확한 답변을 제공하세요.
만약 모든 Context를 다 확인해도 정보가 없다면, "정보가 부족하여 답변할 수 없습니다."를 출력하세요.
---
Context: {context}
---
Question: {question}


In [ ]:
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

def format_docs(docs):
    return "\n---\n".join(["Content: " + doc.page_content for doc in docs])
    # join : 구분자를 기준으로 스트링 리스트를 하나의 스트링으로 연결

rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}

    | prompt
    | llm
    | StrOutputParser()
)

다양한 질문을 통해 결과를 확인해 보겠습니다.

In [17]:
private_questions = [
    '이 회사의 사업은 크게 두 부문으로 나눠집니다. 각각은 어떤 부문입니까?',
    'IT서비스는 몇 개의 분야로 나눠집니까? 각각은 무엇입니까?',
    '2024년과 2025년에 걸쳐, 매출 비중은 어떻게 변화했나요?',
    '2024년 6월말 기준, 연결 재무제표의 유동자산은 얼마인가요?',
    'GPUaaS 서비스가 무엇입니까?',
    '회사의 연구개발 담당조직은 어떤 분야의 핵심 기술을 연구하고 있습니까?',
    '2021-2022년 사이에 있었던 합병 사례를 모두 파악하여, 표로 나타내세요.'
]
result = rag_chain.batch(private_questions)
for i, ans in enumerate(result):
    print(f"Question: {private_questions[i]}")
    print(f"Answer: {ans}")
    print('---')


Question: 이 회사의 사업은 크게 두 부문으로 나눠집니다. 각각은 어떤 부문입니까?
Answer: 이 회사의 사업은 크게 두 부문으로 나누어집니다. 각각 IT서비스 부문과 물류 부문입니다.
---
Question: IT서비스는 몇 개의 분야로 나눠집니까? 각각은 무엇입니까?
Answer: IT서비스는 3개의 분야로 나눠집니다. 각각은 다음과 같습니다.

1. 클라우드 서비스  
2. SI (시스템 통합)  
3. ITO (IT 아웃소싱)
---
Question: 2024년과 2025년에 걸쳐, 매출 비중은 어떻게 변화했나요?
Answer: 2024년과 2025년 매출 비중 변화에 관한 구체적인 2025년 매출 비중 수치는 제공된 Context에 없습니다. 다만, 2024년 매출 비중 현황과 2025년 전망에 관한 일부 정보를 종합하면 다음과 같습니다.

- 2024년 사업부문별 매출 비중은 IT서비스가 약 46.3%, 물류가 약 53.7%입니다.
- IT서비스 매출은 2024년에 클라우드 사업 중심으로 4.8% 증가하였고, IT서비스 내 클라우드 사업 비중이 2023년 31%에서 2024년 36%로 확대되었습니다.
- 2025년 글로벌 IT서비스 시장은 9.0%, 국내 IT서비스 시장은 9.9% 성장할 것으로 전망되며, AI 및 클라우드 기반 디지털 전환 서비스 수요가 증가할 것으로 예상됩니다.
- 2025년 물류 시장은 글로벌 3PL 물류 시장이 연평균 5.6%, 디지털포워딩 시장은 연평균 18.9% 성장할 것으로 전망됩니다.
- 2025년 물류 운임 하락이 예상되나, 디지털 포워딩 서비스 등 차별화된 IT역량 결합으로 물류 서비스 확대 계획이 언급되어 있습니다.

따라서, 2024년 대비 2025년에 IT서비스 부문의 성장률이 물류 부문보다 상대적으로 높을 것으로 예상되나, 정확한 2025년 매출 비중 수치는 제공된 자료에 포함되어 있지 않아 구체적인 변동 수치를 제시하기 어렵습니다.

결론:  
2024년 매출 비중은 IT서비스 약 46.3%, 물

# Multi-Query Retriever   
모호한 쿼리를 검색하는 대신, 다양한 관점에서 Paraphrazing한 쿼리를 사용할 수 있습니다.   
이 때, LLM의 도움을 받을 수 있습니다.

In [19]:
# Multi Query를 확인하기 위한 로깅
import logging

logging.basicConfig()
logging.getLogger('langchain.retrievers.multi_query').setLevel(logging.INFO)

In [ ]:
from langchain_core.prompts import PromptTemplate
from langchain.retrievers.multi_query import MultiQueryRetriever

rewrite_prompt = PromptTemplate(template = """
당신은 삼성SDS의 직원들을 대상으로 하는 챗봇입니다.
기업 공시 문서에 대한 질문이 주어집니다.
'우리 회사' 등의 표현은 회사명으로 변환하세요.     

공시 문서는 '삼성SDS', '삼성에스디에스', 'Samsung SDS' 등의 표현이 혼재된 문서이므로
모든 표현을 하나씩 사용하세요.             

해당 질문에 대한 정보를 검색하기 위해, 벡터 데이터베이스에 입력할 다양한 맥락의 질문을 생성하세요.
질문은 3개 생성하고, 한 줄에 질문 하나씩 출력하세요.
질문을 다각도로 분석하여, 다양한 검색 결과가 나오도록 구성해야 합니다.
---
원본 질문: {question}

""")

multi_query_retriever = MultiQueryRetriever.from_llm(
    retriever=db.as_retriever(),
    llm=llm,
    prompt = rewrite_prompt,
)
# 질문이 주어지면, 질문을 3개로 분할
# 3*K 검색 (12)
# 합집합으로 최종 Context 만들기

In [29]:
len(multi_query_retriever.invoke("2021-2022년 사이에 있었던 합병 사례를 모두 파악하여, 표로 나타내세요."))

INFO:langchain.retrievers.multi_query:Generated queries: ['2021년부터 2022년 사이 삼성SDS의 합병 사례는 무엇인가요?  ', '삼성에스디에스가 2021년부터 2022년 사이에 진행한 기업 합병 내역을 알려주세요.  ', 'Samsung SDS가 2021-2022년 기간 중 합병한 회사와 관련한 공시 문서 내용을 표로 정리해 주세요.']


7

In [35]:
rag_chain = (
    {"context": multi_query_retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

In [ ]:
questions = [
    '이 회사의 사업은 크게 두 부문으로 나눠집니다. 각각은 어떤 부문입니까?',
    'IT서비스는 몇 개의 분야로 나눠집니까? 각각은 무엇입니까?',
    '2024년과 2025년에 걸쳐, 매출 비중은 어떻게 변화했나요?',
    '2024년 6월말 기준, 연결 재무제표의 유동자산은 얼마인가요?',
    'GPUaaS 서비스가 무엇입니까?',
    '회사의 연구개발 담당조직은 어떤 분야의 핵심 기술을 연구하고 있습니까?',
    '2021-2022년 사이에 있었던 합병 사례를 모두 파악하여, 표로 나타내세요.'
]

result = rag_chain.batch(questions)
for i, ans in enumerate(result):
    print(f"Question: {questions[i]}")
    print(f"Answer: {ans}")
    print('---')


### Ensemble Retriever

Ensemble Retriever는 서로 다른 리트리버를 결합하여 순위를 합산합니다.   
주로 Keyword Indexing 기반 검색인 BM25 검색과 Semantic 검색을 합친 Hybrid Search를 사용합니다.

In [40]:
from kiwipiepy import Kiwi

kiwi = Kiwi()
def kiwi_tokenize(text):
    return [token.form for token in kiwi.tokenize(text)]

kiwi_tokenize("2021-2022년 사이에 있었던 합병 사례를 모두 파악하여, 표로 나타내세요.")

['2021-2022',
 '년',
 '사이',
 '에',
 '있',
 '었',
 '던',
 '합병',
 '사례',
 '를',
 '모두',
 '파악',
 '하',
 '어',
 ',',
 '표',
 '로',
 '나타내',
 '세요',
 '.']

In [41]:
from langchain.retrievers import BM25Retriever, EnsembleRetriever

bm25_retriever = BM25Retriever.from_documents(token_chunks, preprocess_func = kiwi_tokenize)
bm25_retriever.k = 5

retriever = db.as_retriever(search_kwargs={"k": 5})

ensemble_retriever = EnsembleRetriever(
    retrievers=[bm25_retriever, retriever], weights=[0.5, 0.5]
)

In [42]:
rag_chain = (
    {"context": ensemble_retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

In [45]:
result = rag_chain.batch(questions)
for i, ans in enumerate(result):
    print(f"Question: {questions[i]}")
    print(f"Answer: {ans}")
    print('---')

Question: 이 회사의 사업은 크게 두 부문으로 나눠집니다. 각각은 어떤 부문입니까?
Answer: 이 회사의 사업은 크게 두 부문으로 나눠지며, 각각은 다음과 같습니다.

1. IT서비스 부문  
2. 물류 부문

IT서비스 부문에서는 클라우드 서비스, 시스템 통합(SI), IT 아웃소싱(ITO) 등 다양한 IT서비스를 제공하고 있으며, 물류 부문에서는 글로벌 물류 전 영역에 걸친 종합 물류 서비스와 디지털 물류 플랫폼을 통한 물류 서비스를 제공합니다.
---
Question: IT서비스는 몇 개의 분야로 나눠집니까? 각각은 무엇입니까?
Answer: IT서비스는 3개의 분야로 나눠집니다. 각각은 클라우드 서비스, SI, ITO입니다.
---
Question: 2024년과 2025년에 걸쳐, 매출 비중은 어떻게 변화했나요?
Answer: 2024년과 2025년에 걸쳐 매출 비중 변화는 다음과 같습니다.

- 2024년 기준으로 사업부문별 매출 비중은 IT서비스가 약 46.3%, 물류가 약 53.7%를 차지하고 있습니다.
- 2024년 IT서비스 매출은 6조 4,014억원(전체 매출의 46.3%)으로 전년 대비 4.8% 증가하였고, 물류 매출은 7조 4,268억원(53.7%)으로 전년 대비 3.6% 증가하였습니다.
- 2025년 국내 IT서비스 시장 규모는 33.2조원으로 2024년 28.5조원 대비 약 16.5% 증가할 것으로 전망되며, 연평균 성장률은 9.9%입니다.
- 2025년 글로벌 IT서비스 시장은 1조 7,315억 달러로 2024년 1조 6,098억 달러 대비 약 7.6% 증가할 것으로 예상됩니다.
- 2025년 글로벌 3PL 물류시장 규모는 1.17조 달러로 2024년 1.59조 달러(2024년 15,879억 달러) 대비 성장하며, 연평균 5.6% 성장할 전망입니다.
- 국내 IT서비스 시장은 2024년 28.5조원에서 2025년 33.2조원으로 성장하며, 물류 시장도 지속 성장할 것으로 보입니다.

요약하면, 2024년과 2025년 모두 

# Reranker
넓은 Retriever 검색 범위를 이용한 뒤, Reranker를 통해 개수를 줄일 수 있습니다.

In [ ]:
# GPU가 있거나, RAM이 충분한 경우에만 아래 코드를 실행해 주세요.
from langchain.retrievers.document_compressors import CrossEncoderReranker
from langchain_community.cross_encoders import HuggingFaceCrossEncoder
from langchain.retrievers import ContextualCompressionRetriever
import torch


# Top 20
rough_retriever = db.as_retriever(search_kwargs={"k": 20})

# 다국어 리랭커 모델 사용하기
model = HuggingFaceCrossEncoder(model_name="BAAI/bge-reranker-v2-m3")

compressor = CrossEncoderReranker(model=model, top_n=5)
# Cross Encoder : 질문과 Chunk를 같이 넣고 처리하는 Transformers 계열 모델
# 점수순으로 Top 5

compression_retriever = ContextualCompressionRetriever(base_compressor=compressor, base_retriever=rough_retriever)
compressed_docs = compression_retriever.invoke("2021-2022년 사이에 있었던 합병 사례를 모두 파악하여, 표로 나타내세요.")

compressed_docs



# Contextual Retrieval    

Claude가 제안한 Contextual Retrieval은 전체 Context를 활용하여     
청크별 헤더를 추가하는 방법입니다.    

In [47]:
# 청크 초기화
token_chunks = token_splitter.split_documents(all_papers)
print(len(token_chunks))

371


In [48]:
context_prompt = ChatPromptTemplate(
    [
        ('user', '''
기업 공시 보고서의 전체 내용과, 그 중 일부 Chunk가 주어집니다.
주어진 Document의 일부인 Chunk에 대해
간결하고 관련성 있는 짧은 설명을 생성하세요.
청크만으로는 명확하지 않으나, 전체를 참고하여 파악할 수 있는 정보를 추가하여
청크의 내용이 더 명확해지도록 하는 2~4문장 길이의 Context를 생성하면 됩니다.
아래의 가이드라인을 참고하세요.

1. 텍스트 부분에서 논의된 주요 주제나 개념을 포함하세요.
2. 문서 전체의 문맥에서 관련 정보나 비교를 언급하세요.
3. 가능한 경우, 이 정보가 문서의 전체적인 주제나 목적과 어떻게 연관되는지를 설명하세요.
4. 중요한 정보를 제공하는 주요 항목과 수치를 포함하세요.
5. 답변은 한국어로 작성합니다.

답변은 간결하게 작성하세요.

# Input Format

- [Document]: `<document> {document} </document>`
- [Chunk]: `<chunk> {chunk} </chunk>`

Context:
        ''')
    ]
)

context_chain = context_prompt | llm | StrOutputParser()


Context가 잘 생성됐는지 확인해 봅니다.

In [49]:
chunk = token_chunks[40]
doc = all_papers[chunk.metadata['index']].page_content
context = context_chain.invoke({'document':doc, 'chunk':chunk.page_content})
print(context)
print('========')
print(chunk.page_content)

해당 청크는 연결재무제표 기준 금융수익과 금융비용 내역, 외환차이의 손익 인식 방식을 상세히 설명하고 있습니다. 또한, 법인세비용의 산출 근거인 연간유효법인세율과 영업활동 현금흐름 조정 내역, 순운전자본 변동사항을 구체적으로 제시하여 회사의 현금흐름 상황을 파악할 수 있도록 합니다. 이 정보는 재무제표 전반의 수익성과 현금흐름 건전성 평가에 중요한 역할을 하며, 외환 변동에 따른 손익 처리 방침도 명확히 하고 있어 재무위험 관리 측면과 연결됩니다.
채 이자비용 2,117,931 3,359,011 기타금융부채 이자비용 10,656,473 20,317,243 기타이자비용 161,768 324,788 외환차손 16,335,257 36,388,025 외화환산손실 5,280,989 13,049,325 전반기 (단위 : 천원) 　 　 장부금액 　 　 공시금액 　 　 3개월 누적 금융수익 64,801,300 158,994,024 금융수익 상각후원가측정금융자 산 이자수익 44,683,046 87,748,899 당기손익-공정가치측정 금융자산 이자수익 384,913 682,037 외환차익 19,169,039 43,928,683 외화환산이익 564,302 26,634,405 금융비용 34,842,543 82,328,882 금융비용 상각후원가측정금융부 채 이자비용 59,031 59,031 기타금융부채 이자비용 13,764,088 18,964,573 기타이자비용 0 0 외환차손 19,219,625 41,423,073 외화환산손실 1,799,799 21,882,205 (2) 연결실체는 외환차이와 관련된 당기손익을 금융수익 및 금융비용으로 인식하고 있습니다. 전자공시시스템 dart.fss.or.kr Page 93 27. 현금흐름표 (연결) 법인세비용은 전체 회계연도에 대해서 예상되는 최선의 가중평균 연간유효법인세율의 추정에 기초 하여 인식하였습니다. 당반기 및 전반기의 예상 평균 연간유효법인세율은 28.1% 및 27.6% 입니다. 연간유효법인세율 당반기 　 　 공시금액 평균유효세율 0.

이제 Context 추가 작업을 수행합니다.

In [ ]:
from tqdm import tqdm

chunk_with_parents=[]

for i, chunk in enumerate(tqdm(token_chunks)):
    doc = all_papers[chunk.metadata['index']].page_content
    chunk_with_parents.append({'document':doc, 'chunk':chunk.page_content})
    # print('\n'+context)
    # print('---')

# 10개씩 실행 (API 여유시 배치 늘리기)
for i in tqdm(range(0, len(token_chunks), 10)):
    contexts = context_chain.batch(chunk_with_parents[i:min(i+10, len(token_chunks))])
    for j in range(i,min(i+10, len(token_chunks))):
        token_chunks[j].page_content = context + '\n\n' + token_chunks[j].page_content

수정된 청크를 이용해, 벡터 데이터베이스를 다시 구성합니다.

In [ ]:
db = Chroma(embedding_function=embeddings,
                           persist_directory="./chroma_Web_ContextualRetrieval",
                           collection_metadata={'hnsw:space':'l2'},
                           collection_name='Report',
                           )

db.add_documents(token_chunks)

Contextual Header를 이용하기 위해, BM25와 Semantic Search를 결합합니다.

In [ ]:
bm25_retriever = BM25Retriever.from_documents(token_chunks, preprocess_func = kiwi_tokenize)
bm25_retriever.k = 5

retriever = db.as_retriever(search_kwargs={"k": 5})

ensemble_retriever = EnsembleRetriever(
    retrievers=[bm25_retriever, retriever], weights=[0.5, 0.5]
)

In [ ]:
rag_chain = (
    {"context": ensemble_retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

result = rag_chain.batch(questions)
for i, ans in enumerate(result):
    print(f"Question: {questions[i]}")
    print(f"Answer: {ans}")
    print('---')